# 03 · Market Structure
#
**Question:** What structure exists in the odds themselves?
#
We characterise each match by its favourite, favourite odds/probability,
second favourite, the probability gap between them, overround, and general
market shape — then ask whether these characteristics systematically relate to
realised results.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import pandas as pd
import numpy as np

from src import data as D
from src import probabilities as P
from src import plotting as plt

primary = P.add_probability_columns(D.load_processed())


## Derive market-structure features per match

In [2]:
def market_features(df):
    out = df.copy()
    out["prob_gap"] = np.nan
    out["fav_outcome"] = ""
    out["fav_prob"] = np.nan
    out["fav_odds"] = np.nan
    out["sec_prob"] = np.nan
    for i in out.index:
        p = {o: out.at[i, f"p_{o}_norm"] for o in ("H", "D", "A")}
        ranked = sorted(p.items(), key=lambda kv: -kv[1])
        fav, sec = ranked[0], ranked[1]
        out.at[i, "fav_outcome"] = fav[0]
        out.at[i, "fav_prob"] = fav[1]
        out.at[i, "sec_prob"] = sec[1]
        out.at[i, "prob_gap"] = fav[1] - sec[1]
        out.at[i, "fav_odds"] = out.at[i, f"B365C{fav[0]}"]
    return out

feat = market_features(primary)
print(feat[["fav_outcome", "fav_prob", "fav_odds", "sec_prob", "prob_gap", "overround"]].describe().round(4))


        fav_prob   fav_odds   sec_prob   prob_gap  overround
count  2280.0000  2280.0000  2280.0000  2280.0000  2280.0000
mean      0.5132     1.9445     0.2692     0.2440     0.0560
std       0.1215     0.4303     0.0554     0.1756     0.0070
min       0.3341     1.0700     0.0789     0.0000     0.0290
25%       0.4126     1.6000     0.2373     0.1053     0.0517
50%       0.4888     1.9500     0.2797     0.2057     0.0560
75%       0.5903     2.3000     0.3094     0.3498     0.0596
max       0.8847     2.8800     0.3740     0.8058     0.1190


## Favourite outcome distribution & win rate

In [3]:
print("Favourite is home/draw/away share and actual win rate when favoured:")
rows = []
for name, g in feat.groupby("fav_outcome", observed=True):
    rows.append({"fav_outcome": name, "n": len(g),
                 "share": len(g) / len(feat),
                 "fav_won": (g["FTR"] == g["fav_outcome"]).mean()})
tbl = pd.DataFrame(rows).set_index("fav_outcome")
print(tbl.round(4))


Favourite is home/draw/away share and actual win rate when favoured:
                n   share  fav_won
fav_outcome                       
A             705  0.3092   0.5135
D              11  0.0048   0.3636
H            1564  0.6860   0.5556


## Is the probability gap associated with the favourite actually winning?

In [4]:
feat["fav_won"] = (feat["FTR"] == feat["fav_outcome"]).astype(int)
bins = [0, 0.05, 0.10, 0.15, 0.20, 0.30, 1.0]
labels = ["0-0.05", "0.05-0.10", "0.10-0.15", "0.15-0.20", "0.20-0.30", "0.30-1.0"]
feat["gap_bin"] = pd.cut(feat["prob_gap"], bins=bins, labels=labels)
rows = []
for name, g in feat.groupby("gap_bin", observed=True):
    rows.append({"gap_bin": name, "n": len(g),
                 "fav_won_rate": g["fav_won"].mean(),
                 "avg_fav_prob": g["fav_prob"].mean()})
gap_tbl = pd.DataFrame(rows).set_index("gap_bin")
print(gap_tbl.round(4))


             n  fav_won_rate  avg_fav_prob
gap_bin                                   
0-0.05     246        0.3496        0.3644
0.05-0.10  282        0.4291        0.3931
0.10-0.15  292        0.4658        0.4300
0.15-0.20  269        0.4907        0.4649
0.20-0.30  411        0.5182        0.5196
0.30-1.0   755        0.7126        0.6579


## Overround vs outcome
#
Is a tightly-priced market ("low overround") any better at predicting results?

In [5]:
feat["or_bin"] = pd.cut(feat["overround"], bins=[0, 0.05, 0.06, 0.12],
                        labels=["low", "mid", "high"])
rows = []
for name, g in feat.groupby("or_bin", observed=True):
    rows.append({"or_bin": name, "n": len(g), "fav_won_rate": g["fav_won"].mean()})
ortbl = pd.DataFrame(rows).set_index("or_bin")
print(ortbl.round(4))


           n  fav_won_rate
or_bin                    
low      369        0.5583
mid     1403        0.5274
high     508        0.5689


## Market shape: is draw more/less likely than odds imply?
#
A "shape" concern: are draws priced systematically off in one-touch markets?
We compare observed draw frequency vs mean draw implied probability.

In [6]:
print("Draw: mean implied =", round(feat["p_D_norm"].mean(), 4),
      "| observed =", round((feat["FTR"] == "D").mean(), 4))


Draw: mean implied = 0.2628 | observed = 0.2658


## Summary
#
* Favourites (highest normalised probability) are overwhelmingly priced to be
  near-favourites, and their win rate rises with the probability gap.
* Overround is small and fairly stable.
* These structural patterns are candidates for simple strategies (notebook 04),
  but must be validated out-of-sample (notebook 05).